# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Same mid-panel dev window as every notebook in this repo (ML-04 through ML-10) --
# never the sealed June 2026 sample (fact_content_daily_performance_sample).
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Connected. Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")

# --- Rebuild the identical feature vector + label as ML-05/07/08/09/10 ---
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()
feature_frame["ctr_prev30"] = (feature_frame["clk_prev30"] / feature_frame["imp_prev30"]).fillna(0)

content_schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
EXCLUDE_LIKE = ("client", "hash", "id", "profile", "account", "flag", "score")
candidate_cols = [
    c for c in content_schema.loc[content_schema["column_type"] == "VARCHAR", "column_name"]
    if not any(bad in c.lower() for bad in EXCLUDE_LIKE)
]
cat_features = []
for col in candidate_cols:
    n_distinct = con.sql(f"SELECT COUNT(DISTINCT {col}) FROM {TABLES['dim_content']}").fetchone()[0]
    if 1 < n_distinct <= 15:
        cat_features.append(col)

cols_sql = ", ".join(cat_features)
content_meta = con.sql(f"SELECT content_hash_id, {cols_sql} FROM {TABLES['dim_content']}").df()
feature_frame = feature_frame.merge(content_meta, on="content_hash_id", how="left")
for col in cat_features:
    feature_frame[col] = feature_frame[col].fillna("unknown")
feature_frame = pd.get_dummies(feature_frame, columns=cat_features, prefix=cat_features)

march = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()
data = feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)

feature_cols = [c for c in feature_frame.columns if c not in ("client_hash_id", "content_hash_id")]
print(f"\n{len(data):,} content items, {len(feature_cols)} feature columns, base rate {data['is_declining'].mean():.3f}")
print("Matches ML-05/07/08/09/10's 146,253 rows / 0.285 base rate if the pipeline is consistent.")

# --- Population scoping, for the Data section (ML-09's fix: report as 3 separate steps) ---
n_warehouse_total = con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_content']}").fetchone()[0]
n_march_scoped = len(feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner"))
n_after_filter = len(data)

# --- Baseline rule, identical to ML-07/08/10 ---
content_dates = (con.sql(f"SELECT content_hash_id, content_updated_date FROM {TABLES['dim_content']}")
                  .df().drop_duplicates("content_hash_id").set_index("content_hash_id")["content_updated_date"])
data["content_updated_date"] = data["content_hash_id"].map(content_dates)
update_date = pd.to_datetime(data["content_updated_date"])
month_start_ts = pd.Timestamp(MONTH_START)
valid_update = update_date <= month_start_ts
data["days_since_update"] = np.where(valid_update, (month_start_ts - update_date).dt.days, np.nan)
n_future_dated = int((~valid_update & update_date.notna()).sum())

data["position_bucket"] = pd.cut(
    data["avg_position_prev30"], bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)
ctr_by_position = data.dropna(subset=["position_bucket"]).groupby("position_bucket", observed=True).agg(
    total_clicks=("clk_prev30", "sum"), total_impressions=("imp_prev30", "sum")
)
ctr_by_position["expected_ctr"] = ctr_by_position["total_clicks"] / ctr_by_position["total_impressions"]
data["expected_ctr"] = data["position_bucket"].map(ctr_by_position["expected_ctr"]).astype(float)

VISIBILITY_FLOOR = 100  # below this, a CTR reading is too noisy to trust
visible = data["imp_prev30"] >= VISIBILITY_FLOOR
has_position = data["position_bucket"].notna()
ctr_gap = (data["expected_ctr"] - data["ctr_prev30"]).clip(lower=0)
staleness_multiplier = 1 + data["days_since_update"].fillna(0) / 365  # unknown -> neutral (1x)
data["baseline_score"] = np.where(visible & has_position, ctr_gap * data["imp_prev30"] * staleness_multiplier, 0.0)
data["ctr_gap"] = ctr_gap

# --- Client-grouped split, identical params to ML-05/07/08/09/10 -> identical 29/13/0 partition ---
from sklearn.model_selection import GroupShuffleSplit

X_full = data[feature_cols].fillna(0)
y_full = data["is_declining"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_full, y_full, groups=groups))
X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]
n_train_clients = groups.iloc[train_idx].nunique()
n_test_clients = groups.iloc[test_idx].nunique()
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
test_base_rate = y_test.mean()

# --- Train Logistic Regression + Random Forest on the SAME split, identical to ML-08 ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

lr = LogisticRegression(max_iter=2000, random_state=42).fit(X_train, y_train)
lr_scores_test = lr.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
rf_scores_test = rf.predict_proba(X_test)[:, 1]

baseline_scores_test = data["baseline_score"].iloc[test_idx]

def precision_at_k(scores, labels, k=50):
    ranked = pd.DataFrame({"score": np.asarray(scores), "label": np.asarray(labels)}).sort_values("score", ascending=False)
    return ranked.head(k)["label"].mean()

def summarize(name, scores, labels, base_rate):
    p50 = precision_at_k(scores, labels, k=50)
    return {
        "method": name, "n_test": len(labels), "base_rate": round(base_rate, 3),
        "precision_at_50": round(p50, 3),
        "lift": round(p50 / base_rate, 2) if base_rate > 0 else np.nan,
        "auc": round(roc_auc_score(labels, scores), 3),
    }

comparison = pd.DataFrame([
    summarize("Rule baseline", baseline_scores_test, y_test, test_base_rate),
    summarize("Logistic Regression", lr_scores_test, y_test, test_base_rate),
    summarize("Random Forest", rf_scores_test, y_test, test_base_rate),
]).set_index("method")

print(f"\nClient-grouped split: {n_train_clients} train / {n_test_clients} test clients, overlap={len(overlap)}")
print(f"Held-out test set: n={len(test_idx):,}, base rate={test_base_rate:.3f}")
print(f"\n{comparison}")

## 1. Question

*The research question and the decision it supports.*

**Question:** which pages should the content team review first — for refresh, expansion,
protection, pruning, or monitoring? (Lane 2, `w01_research_question.ipynb` /
`w02_ml_task_framing.ipynb`.)

**Decision it supports:** a sorted queue an editorial team works down from the top each refresh
cycle. Mechanically this is a ranking problem solved with a binary classifier: a model (or, as
it turned out, a transparent rule) estimates the probability a page is declining, and that
probability becomes the sort key.

**Success metric:** Precision@50 — of the top 50 pages the queue puts first, what fraction are
truly declining? k=50 was chosen because that's roughly what an editorial team can act on in one
cycle; ranking the other ~146,000 pages accurately doesn't matter if nobody will ever reach
them. Every score in this notebook is reported next to its base rate, never alone.

In [ ]:
print("Lane 2 -- Refresh / Content Opportunity Scoring")
print("Question: which pages should the content team review first?")
print("Success metric: Precision@50 (reported next to its base rate, never alone)")
print(f"\nThis run's population: {len(data):,} content items, base rate {data['is_declining'].mean():.3f}")

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse` on Hugging Face — gated, ~79M rows across 5 tables,
104 clients, `2025-01-27` to `2026-06-30`. Never downloaded; queried in place with DuckDB over
`hf://...` parquet files. Tables used here: `fact_content_daily_performance` (the daily GSC
panel this notebook's features and label come from) and `dim_content` (content metadata, and
the baseline rule's staleness input). `dim_clients` and `fact_content_query_90d` weren't needed
for this lane.

**Date windows:** a mid-panel development month, `2026-03`, chosen so a real 30-day "before"
window (`2026-01-30` to `2026-03-01`) exists on both sides — never the sealed final month
(`2026-06`, `fact_content_daily_performance_sample`), which stays untouched.

**Population narrowing** — three separate steps, reported separately rather than collapsed into
one number (a fix ML-09 made after an earlier version of this repo conflated them):
1. warehouse-wide `dim_content` (all clients, all time)
2. scoped to the March-2026 slice (a client/date join, not a filter choice)
3. after requiring `imp_prev30 > 0` — a prev30-only, pre-label-window filter (items with no
   prior visibility can't get a trustworthy CTR reading)

Exact counts for this run are printed below.

**What's excluded and why:**
- Raw same-period (March) impressions/clicks are never a feature — they determine the label
  itself, the canonical leakage trap this repo tracks.
- Product/decision flags (score-, flag-, priority-like columns) are never features — using an
  existing system's decision as an input just re-learns that system, not the world
  (`skills/hunting-leakage-and-validating/SKILL.md`).
- `client_hash_id`/`content_hash_id` are for grouping and joining only, never features.
- No client names, domains, URLs, or raw queries appear anywhere in this notebook or its
  outputs — only the pseudonymized hashes (`DATA_USE.md`).

In [ ]:
print(f"Warehouse-wide dim_content: {n_warehouse_total:,} items")
print(f"March-2026-scoped (client/date join): {n_march_scoped:,} items")
print(f"After imp_prev30 > 0 filter (this notebook's actual population): {n_after_filter:,} items")
print(f"\nTables used: fact_content_daily_performance (features + label), dim_content (metadata + baseline rule inputs)")
print(f"Dev window: prev30 [{PREV30_START}, {PREV30_END_EXCL}) | label window [{MONTH_START}, {MONTH_END_EXCL})")
print("Sealed June 2026 sample (fact_content_daily_performance_sample): never queried in this notebook.")

# No-go check: confirm nothing client-identifying is sitting in this notebook's own columns
BANNED_IN_DATA = ("title", "url", "domain", "query", "name", "email")
hits = [c for c in data.columns if any(bad in c.lower() for bad in BANNED_IN_DATA)]
print(f"\nClient-identifying-looking columns in `data`: {hits or 'none'}")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Unit of analysis:** one content item (a page) per client, scored independently.

**Label:** `is_declining = (imp_march < 0.8 x imp_prev30)` — impressions dropped more than 20%
month-over-month, for items with at least some prior visibility (`imp_prev30 > 0`). A proxy for
"needs review," not a claim about ranking causes.

**Features:** 5 numeric prev30 aggregates (`imp_prev30`, `clk_prev30`, `avg_position_prev30`,
`ctr_prev30`, `active_days_prev30`) + 4 low-cardinality categoricals discovered live on
`dim_content` (`content_type`, `competition_level`, `main_intent`, `model_used`), one-hot
encoded. Every feature is knowable strictly before the label window — confirmed below.

**Baseline (the rule this repo defends until something beats it):** `ctr_gap x imp_prev30 x
staleness_multiplier`, where `ctr_gap` is the shortfall between a page's own CTR and its
position tier's expected CTR — said in plain words first, per
`skills/building-baselines/SKILL.md`: "a page is worth reviewing if it's visible, underperforming
its position's expected click rate, and has gone a while since its last update." Staleness stays
a near-neutral 1x multiplier for most rows because `content_updated_date` is a current-state
field, not frozen at `MONTH_START` — most items can't be scored for pre-March staleness at all,
named plainly rather than hidden (ML-07).

**Validation design — client-grouped, not random:** pages from the same client share structure
(template, niche, traffic tier), so a random split lets a model partly memorize "this client's
pages behave like X" instead of learning something general. `GroupShuffleSplit(test_size=0.3,
random_state=42)` grouped on `client_hash_id` reproduces the same 29-train/13-test client
partition every notebook in this repo has used since ML-05, confirmed below. Not time-aware on
top of that: prev30/March is this slice's only time dimension, and a true chronological holdout
is what the sealed June sample is for, not routine model comparison.

**Leakage checks** (full detail in ML-09, `w08_validation_audit.ipynb`; the confession test is
re-confirmed fresh below): the timeline (no feature query touches `report_date >= MONTH_START`),
no product/decision flags in the feature list, the population filter disclosed above, the split
grouped by client, base rate printed next to every metric, top feature importances
sanity-checked (no single feature towers, ML-08), metrics always computed out-of-fold. A naive
random split was also tested against this grouped one (ML-09): it inflates Precision@50 by a
real, explainable margin — client memorization, not signal — which is exactly why the grouped
split is the one this paper reports.

In [ ]:
from sklearn.model_selection import train_test_split

def quick_auc(cols, df):
    X = df[cols].fillna(0)
    y = df["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

# --- Confession test: does adding the label-derived column collapse AUC toward 1.0? ---
honest_auc = quick_auc(feature_cols, data)
data["imp_march_leak"] = data["imp_march"]
leaky_auc = quick_auc(feature_cols + ["imp_march_leak"], data)
data = data.drop(columns=["imp_march_leak"])
restored_auc = quick_auc(feature_cols, data)
print(f"Confession test -- honest AUC: {honest_auc:.3f} | +raw March impressions: {leaky_auc:.3f} | restored: {restored_auc:.3f}")

print(f"\nTimeline: prev30 = [{PREV30_START}, {PREV30_END_EXCL}) | label window = [{MONTH_START}, {MONTH_END_EXCL}) -- no overlap")
print(f"Client-grouped split: {n_train_clients} train / {n_test_clients} test clients, overlap={len(overlap)}")
print(f"Held-out base rate: {test_base_rate:.3f} (n_test={len(test_idx):,})")

FLAG_LIKE = ("score", "flag", "priority", "risk", "declin", "trend", "recommend", "status")
hits = [c for c in feature_cols if any(bad in c.lower() for bad in FLAG_LIKE)]
print(f"Product/decision-flag-like columns in feature_cols: {hits or 'none'}")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

All three rows below come from one split, one metric, computed together in this notebook —
the same comparison ML-08 (`w07_model.ipynb`) originally ran, re-run here for the paper so the
numbers and the chart come from this run specifically:

| Method | Precision@50 (ML-08's original run) | Lift | AUC |
|---|---|---|---|
| Rule baseline | 0.540 | 1.69x | 0.460 |
| Logistic Regression | 0.420 | 1.31x | 0.629 |
| Random Forest | 0.520 | 1.63x | 0.626 |

This run's exact numbers are printed below — small run-to-run drift in the Random Forest row is
expected and documented (see Limitations).

**The rule baseline wins.** A striking side detail: its AUC (0.460) is *below* random, while
both models sit around 0.63 — the rule is a poor whole-queue ranker, sharply tuned for exactly
the top 50, which is this repo's one defensible metric. AUC and Precision@K measure different
things; the stable claim stays Precision@50, not AUC.

**Naive split vs. honest split (ML-09):** the same Random Forest scored under a naive random
70/30 split instead of the client-grouped one came back materially higher — client memorization
inflating the number, not real skill. Every number in the table above and below is the honest,
client-grouped one.

In [ ]:
print(comparison)

# --- New chart: method comparison, Precision@50 vs base rate ---
import matplotlib.pyplot as plt

os.makedirs("work/figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 3.5))
methods = comparison.index.tolist()
precisions = comparison["precision_at_50"].tolist()
ax.bar(methods, precisions, color=["#426B69", "#8C6BB1", "#4E79A7"])
ax.axhline(test_base_rate, color="#B00020", linestyle="--", linewidth=1, label=f"base rate ({test_base_rate:.3f})")
ax.set_ylabel("Precision@50 (held-out)")
ax.set_title("Method comparison: rule baseline vs. model")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("work/figures/method_comparison.png", dpi=150)
plt.close()
print("\nWrote work/figures/method_comparison.png")

## 5. Limitations

*What this work cannot claim.*

- **Staleness is uninformative for most of the queue.** `content_updated_date` is a
  current-state field, not point-in-time — most items can't be scored for pre-March staleness
  at all, so the rule's staleness term is a near-neutral multiplier for the bulk of the
  population (ML-07).
- **One client-grouped split, one month, validated.** The 0.540/1.69x number comes from 13
  held-out clients scored against the March-2026 slice. A new client, or a different month,
  hasn't been checked and needs its own validation before this queue is trusted for it.
- **Validated only at k=50.** Ranking below roughly rank 50-100 hasn't been checked against a
  held-out label with the same confidence.
- **The rule's whole-queue AUC is below random (0.460).** It's a legitimately narrow tool —
  sharply tuned for the top 50, not a general-purpose ranker of the full ~146K-item queue.
- **Observational, not causal.** Nothing here claims a refresh *causes* recovery — the data is
  a cross-sectional snapshot, not an experiment.
- **The Random Forest number is library-version sensitive.** Re-running the identical
  model/split across Colab sessions has produced Precision@50 anywhere from about 0.44 to 0.52
  (ML-09) — the stable claim is the model landing near, but not beating, the baseline, not the
  third decimal.
- **The population filter itself is a choice.** Only items with `imp_prev30 > 0` — some prior
  visibility — get scored at all; that choice is disclosed here, not hidden.

In [ ]:
n_unknown_staleness = (~data["days_since_update"].notna()).sum()
print(f"Staleness known for {data['days_since_update'].notna().mean()*100:.1f}% of scored items "
      f"({n_unknown_staleness:,} of {len(data):,} unknown)")
print(f"Validation covers {n_train_clients} train / {n_test_clients} test clients, "
      f"{n_train_clients + n_test_clients} total -- a client outside these has not been validated against")
print(f"Validation month: {MONTH_START[:7]} only")
print(f"\nRule baseline AUC this run: {comparison.loc['Rule baseline', 'auc']:.3f} (below 0.500 -- a narrow, not general, tool)")
print(f"Population filter: {n_after_filter:,} of {n_march_scoped:,} March-scoped items pass imp_prev30 > 0 "
      f"({n_after_filter / n_march_scoped:.1%})")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Full design and reasoning in ML-10 (`w09_action_playbook.ipynb`); recomputed fresh here on this
run's `data` so the numbers below match this notebook exactly.

**Ranker:** the rule baseline stays primary — Section 4 just re-confirmed it beats the Random
Forest, so nothing changed that decision. The Random Forest's probability rides along as a
secondary `model_agrees` corroboration flag only, never used to re-rank
(`skills/building-baselines/SKILL.md`: keep the baseline frozen once model work starts).

**Confidence tiers:** high/medium/low by baseline-score quantile (p80/p50) within the scored
population. **Action:** `refresh` when the baseline score is positive, `monitor` otherwise.
**Reason codes:** CTR-gap size (`large_ctr_gap`/`small_ctr_gap`), staleness state
(`stale_180d_plus`/`recently_updated`/`update_date_unknown`), and `model_agrees` when the RF
probability is also >=0.5 on the same row.

**No-go list:** never auto-publish/deprioritize from this queue's action label; never treat
`refresh` as a promise of recovery; never re-rank by `rf_probability`; never export anything
beyond `client_hash_id`/`content_hash_id`.

In [ ]:
scored_mask = data["baseline_score"] > 0
data["rf_probability"] = rf.predict_proba(X_full)[:, 1]

high_threshold = data.loc[scored_mask, "baseline_score"].quantile(0.8)
medium_threshold = data.loc[scored_mask, "baseline_score"].quantile(0.5)

def confidence_label(score):
    if score >= high_threshold:
        return "high"
    if score >= medium_threshold:
        return "medium"
    return "low"

data["confidence"] = np.where(scored_mask, data["baseline_score"].apply(confidence_label), "not_scored")

def reason_codes(row):
    if row["baseline_score"] <= 0:
        return "not_scored"
    codes = []
    gap_pct = row["ctr_gap"] * 100
    codes.append("large_ctr_gap" if gap_pct >= 0.30 else "small_ctr_gap")
    if pd.isna(row["days_since_update"]):
        codes.append("update_date_unknown")
    elif row["days_since_update"] >= 180:
        codes.append("stale_180d_plus")
    else:
        codes.append("recently_updated")
    if row["rf_probability"] >= 0.5:
        codes.append("model_agrees")
    return "|".join(codes)

data["reason_codes"] = data.apply(reason_codes, axis=1)
data["action"] = np.where(scored_mask, "refresh", "monitor")

final_queue = data.sort_values("baseline_score", ascending=False).reset_index(drop=True)
final_queue.insert(0, "rank", final_queue.index + 1)

print("Action counts:")
print(final_queue["action"].value_counts())
print("\nConfidence tier counts:")
print(final_queue["confidence"].value_counts())
print("\nTop 5 rows:")
print(final_queue.head(5)[["rank", "action", "confidence", "reason_codes", "baseline_score", "rf_probability"]])

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three charts, all regenerated fresh in this notebook so they match the numbers printed above
exactly: the method-comparison bar chart (Section 4), and the action/confidence mix charts
(Section 6, same design as ML-10). Written to `work/figures/`.

In [ ]:
action_counts = final_queue["action"].value_counts()
plt.figure(figsize=(5, 3))
action_counts.plot(kind="bar", color="#426B69")
plt.title("Action mix")
plt.ylabel("content items")
plt.tight_layout()
plt.savefig("work/figures/action_mix.png", dpi=150)
plt.close()

confidence_counts = final_queue["confidence"].value_counts().reindex(
    ["high", "medium", "low", "not_scored"], fill_value=0
)
plt.figure(figsize=(5, 3))
confidence_counts.plot(kind="bar", color="#6F4E7C")
plt.title("Confidence mix")
plt.ylabel("content items")
plt.tight_layout()
plt.savefig("work/figures/confidence_mix.png", dpi=150)
plt.close()

print("Artifacts for the paper:")
print("- work/figures/method_comparison.png (Section 4)")
print("- work/figures/action_mix.png (this run)")
print("- work/figures/confidence_mix.png (this run)")
print("\nKey numbers for the paper text:")
print(comparison)
print(f"\nAction counts: {final_queue['action'].value_counts().to_dict()}")
print(f"Confidence counts: {final_queue['confidence'].value_counts().to_dict()}")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
